## Why this analysis

Goma sits at the intersection of two fragility logics that rarely overlap in the same city. To the south is Lake Kivu; to the north is Mount Nyiragongo, an active volcano whose 2021 eruption destroyed parts of the city and displaced tens of thousands. To the west and north lie zones of recurring armed conflict tied to the M23 insurgency and decades of regional war, which has produced waves of internally displaced people now living in camps around the city's edges.

The atlas question applies again: where do people live, where are the health facilities, and where is the gap? Goma adds a third dimension absent from the other cities: the population is partly volcanic-hazard exposed and partly conflict-displaced.

In [1]:
# Goma — Service Density Analysis
# Part of the Atlas of Urban Fragility

import osmnx as ox
import geopandas as gpd
import pandas as pd
import folium
import h3
from shapely.geometry import Polygon, Point
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Sanity check
print(f"osmnx:      {ox.__version__}")
print(f"geopandas:  {gpd.__version__}")
print(f"folium:     {folium.__version__}")
print(f"h3:         {h3.__version__}")

# Project paths
PROJECT_ROOT = Path.cwd().parent  # we're in atlas/, so parent is heissj.github.io/
DATA_DIR = PROJECT_ROOT / "data" / "goma"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"\nData directory: {DATA_DIR}")

# Verify we're on the right Python
import sys
print(f"\nPython: {sys.executable}")

osmnx:      2.1.0
geopandas:  1.1.3
folium:     0.20.0
h3:         4.4.1

Data directory: /Users/sajjadsharifi/UrbanProjects/heissj.github.io/data/goma

Python: /opt/homebrew/Caskroom/miniforge/base/envs/urban/bin/python


In [2]:
# Area of interest: Goma and surrounding settlements
# Covers the city proper plus northern IDP-hosting areas and the area south
# toward Lake Kivu. Mount Nyiragongo lies just north of this bbox.
AREA = {
    'name': "Goma (city and northern periphery)",
    'bbox': {
        'south': -1.72,
        'north': -1.60,
        'west':  29.18,
        'east':  29.30,
    }
}

center_lat = (AREA['bbox']['south'] + AREA['bbox']['north']) / 2
center_lon = (AREA['bbox']['west']  + AREA['bbox']['east'])  / 2

print(f"Area:   {AREA['name']}")
print(f"Center: {center_lat:.4f}°, {center_lon:.4f}°E")
print(f"Span:   {AREA['bbox']['north']-AREA['bbox']['south']:.3f}° lat × "
      f"{AREA['bbox']['east']-AREA['bbox']['west']:.3f}° lon")

Area:   Goma (city and northern periphery)
Center: -1.6600°, 29.2400°E
Span:   0.120° lat × 0.120° lon


In [3]:
# Quick map to confirm the area looks right
m = folium.Map(location=[center_lat, center_lon], zoom_start=12,
               tiles='OpenStreetMap')

# Draw the bounding box
b = AREA['bbox']
folium.Rectangle(
    bounds=[[b['south'], b['west']], [b['north'], b['east']]],
    color='#cc0000', weight=2,
    fill=True, fill_opacity=0.1,
    tooltip=AREA['name']
).add_to(m)

m

## Where are the health facilities?

OpenStreetMap coverage of Goma improved meaningfully after the 2021 Nyiragongo eruption, when HOTOSM and humanitarian agencies mapped lava-flow zones and damaged areas. Health facility coverage remains uneven, especially for smaller clinics in the IDP camps that have appeared since 2022. This query captures hospitals, clinics, doctors' offices, pharmacies, dentists, and anything tagged as healthcare.

In [4]:
# OSM tags for health facilities — covers hospitals, clinics, pharmacies,
# doctors' offices, and anything tagged as healthcare
tags = {
    'amenity': ['hospital', 'clinic', 'doctors', 'pharmacy'],
    'healthcare': True,
}

# osmnx 2.x bbox format: (west, south, east, north)
bbox_tuple = (b['west'], b['south'], b['east'], b['north'])

print(f"Querying OpenStreetMap for health facilities in {AREA['name']}...")
facilities = ox.features.features_from_bbox(bbox=bbox_tuple, tags=tags)
print(f"\nReturned {len(facilities)} features")
print(f"\nFirst few rows:")
facilities.head()

Querying OpenStreetMap for health facilities in Goma (city and northern periphery)...

Returned 342 features

First few rows:


geometry   amenity  \
element id                                                
node    5811621863  POINT (29.23915 -1.68409)  pharmacy   
        5811621874  POINT (29.23382 -1.68566)  pharmacy   
        5811621875   POINT (29.23412 -1.6854)  pharmacy   
        6095244622  POINT (29.21794 -1.67773)   doctors   
        6123220045  POINT (29.19062 -1.63895)    clinic   

                                          name is_in:health_area  \
element id                                                         
node    5811621863           Pharmacie Muvunga               NaN   
        5811621874         Pharmacie Diocésain               NaN   
        5811621875  Pharmacie Arche de l'unité               NaN   
        6095244622     Centre de santé Katindo           Katindo   
        6123220045      Centre de santé Hebron            Hebron   

                   is_in:health_zone     source source:date check_date  \
element id                                                               
node    5811621863               NaN        NaN         NaN        NaN   
        5811621874               NaN        NaN         NaN        NaN   
        5811621875               NaN        NaN         NaN        NaN   
        6095244622              Goma  MSFsurvey        2018        NaN   
        6123220045         Karisimbi  MSFsurvey     2019-01        NaN   

                   healthcare fixme  ... name:fr name:rw wikidata  \
element id                           ...                            
node    5811621863        NaN   NaN  ...     NaN     NaN      NaN   
        5811621874        NaN   NaN  ...     NaN     NaN      NaN   
        5811621875        NaN   NaN  ...     NaN     NaN      NaN   
        6095244622        NaN   NaN  ...     NaN     NaN      NaN   
        6123220045        NaN   NaN  ...     NaN     NaN      NaN   

                   healthcare:speciality barrier layer building description  \
element id                                                                    
node    5811621863                   NaN     NaN   NaN      NaN         NaN   
        5811621874                   NaN     NaN   NaN      NaN         NaN   
        5811621875                   NaN     NaN   NaN      NaN         NaN   
        6095244622                   NaN     NaN   NaN      NaN         NaN   
        6123220045                   NaN     NaN   NaN      NaN         NaN   

                   website wikimedia_commons  
element id                                    
node    5811621863     NaN               NaN  
        5811621874     NaN               NaN  
        5811621875     NaN               NaN  
        6095244622     NaN               NaN  
        6123220045     NaN               NaN  

[5 rows x 42 columns]

In [5]:
# What kinds of facilities are in the data?
print("Amenity types:")
print(facilities['amenity'].value_counts() if 'amenity' in facilities.columns else "(no amenity column)")

print("\nHealthcare types:")
print(facilities['healthcare'].value_counts() if 'healthcare' in facilities.columns else "(no healthcare column)")

print(f"\nGeometry types:")
print(facilities.geometry.geom_type.value_counts())

Amenity types:
amenity
clinic         167
health_post     78
hospital        52
pharmacy        20
doctors         17
Name: count, dtype: int64

Healthcare types:
healthcare
alternative        45
nurse              35
hospital           25
clinic             13
centre              8
yes                 5
laboratory          2
psychotherapist     1
Name: count, dtype: int64

Geometry types:
Point      307
Polygon     35
Name: count, dtype: int64


In [6]:
# Convert polygon geometries to their centroids; points stay as points
# Project to Web Mercator first for accurate centroid math, then back to WGS84
facilities_proj = facilities.to_crs(epsg=3857)
facilities_proj['geometry'] = facilities_proj.geometry.centroid
facilities_pts = facilities_proj.to_crs(epsg=4326)

print(f"{len(facilities_pts)} facilities, all as points now")
print(f"Geometry types: {facilities_pts.geometry.geom_type.value_counts().to_dict()}")

342 facilities, all as points now
Geometry types: {'Point': 342}


In [7]:
# H3 resolution 8 → hexagons roughly 0.5 km across
# (Good resolution for a camp-scale analysis; res 7 = too coarse, res 9 = too fine)
H3_RESOLUTION = 8

# Define our area as an H3 polygon (vertices in lat, lng order)
vertices = [
    (b['south'], b['west']),
    (b['north'], b['west']),
    (b['north'], b['east']),
    (b['south'], b['east']),
]
h3_poly = h3.LatLngPoly(vertices)

# Get all H3 cells inside this polygon
hex_ids = h3.h3shape_to_cells(h3_poly, H3_RESOLUTION)
print(f"Generated {len(hex_ids)} H3 hexes at resolution {H3_RESOLUTION}")
print(f"First few hex IDs: {hex_ids[:3]}")

Generated 212 H3 hexes at resolution 8
First few hex IDs: ['886adc49abfffff', '886adeb30dfffff', '886adeb025fffff']


In [8]:
# Build a GeoDataFrame where each row is one hexagon
hex_records = []
for h in hex_ids:
    # h3.cell_to_boundary returns list of (lat, lng) tuples
    # Shapely expects (lng, lat) order
    boundary = h3.cell_to_boundary(h)
    poly = Polygon([(lng, lat) for lat, lng in boundary])
    hex_records.append({'h3_id': h, 'geometry': poly})

hexes = gpd.GeoDataFrame(hex_records, crs='EPSG:4326')

# Spatial join: which hex contains each facility?
joined = gpd.sjoin(facilities_pts, hexes, how='left', predicate='within')

# Count facilities per hex
counts = joined.groupby('h3_id').size().rename('facility_count')
hexes = hexes.merge(counts, on='h3_id', how='left')
hexes['facility_count'] = hexes['facility_count'].fillna(0).astype(int)

# Quick stats
print(f"Total hexes:                {len(hexes)}")
print(f"Hexes with ≥1 facility:     {(hexes['facility_count'] > 0).sum()}")
print(f"Hexes with ≥3 facilities:   {(hexes['facility_count'] >= 3).sum()}")
print(f"Max facilities in one hex:  {hexes['facility_count'].max()}")
print(f"\nDistribution of facility counts:")
print(hexes['facility_count'].value_counts().sort_index())

Total hexes:                212
Hexes with ≥1 facility:     61
Hexes with ≥3 facilities:   40
Max facilities in one hex:  20

Distribution of facility counts:
facility_count
0     151
1      13
2       8
3       5
4       4
5       4
6       8
7       2
9       4
10      4
11      4
12      1
13      1
17      2
20      1
Name: count, dtype: int64


In [9]:
# Set up the map
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles='OpenStreetMap'
)

# Style function: color hexes by facility count
def style_fn(feature):
    count = feature['properties']['facility_count']
    if count == 0:
        return {'fillColor': '#ffffff', 'color': '#cccccc',
                'weight': 0.3, 'fillOpacity': 0.0}
    elif count == 1:
        return {'fillColor': '#9ecae1', 'color': '#4292c6',
                'weight': 0.6, 'fillOpacity': 0.6}
    elif count == 2:
        return {'fillColor': '#4292c6', 'color': '#2171b5',
                'weight': 0.6, 'fillOpacity': 0.7}
    else:  # 3+
        return {'fillColor': '#08519c', 'color': '#08306b',
                'weight': 0.8, 'fillOpacity': 0.85}

# Add the hexes
folium.GeoJson(
    hexes.to_json(),
    name='Health facility density (per hex)',
    style_function=style_fn,
    tooltip=folium.GeoJsonTooltip(
        fields=['facility_count'],
        aliases=['Health facilities:']
    )
).add_to(m)

# Add individual facility points on top for context
for _, row in facilities_pts.iterrows():
    name = (row.get('name:en') or row.get('name')
            or str(row.get('amenity', 'Health facility')))
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3, color='#222', weight=1,
        fill=True, fillColor='white', fillOpacity=1.0,
        tooltip=name
    ).add_to(m)

# Add a simple legend (fixed-position HTML overlay)
legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px;
            background-color: white; border: 1px solid #999;
            padding: 10px 14px; font-family: Arial, sans-serif;
            font-size: 12px; z-index: 9999;
            box-shadow: 0 2px 6px rgba(0,0,0,0.15); line-height: 1.6;">
    <div style="font-weight: bold; margin-bottom: 6px;">
        Health facilities per hex
    </div>
    <div><span style="background:#08519c;opacity:0.85;width:18px;height:14px;
                      display:inline-block;vertical-align:middle;margin-right:6px;"></span>3 or more</div>
    <div><span style="background:#4292c6;opacity:0.7;width:18px;height:14px;
                      display:inline-block;vertical-align:middle;margin-right:6px;"></span>2</div>
    <div><span style="background:#9ecae1;opacity:0.6;width:18px;height:14px;
                      display:inline-block;vertical-align:middle;margin-right:6px;"></span>1</div>
    <div><span style="background:white;border:1px solid #ccc;width:18px;height:14px;
                      display:inline-block;vertical-align:middle;margin-right:6px;"></span>0</div>
    <div style="border-top:1px solid #eee;margin-top:6px;padding-top:4px;font-size:11px;color:#666;">
        Hex resolution: H3 res 8 (~0.5 km)
    </div>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Display
m

In [10]:
output_path = PROJECT_ROOT / "atlas" / "goma-map.html"
m.save(str(output_path))
print(f"Map saved to: {output_path}")

Map saved to: /Users/sajjadsharifi/UrbanProjects/heissj.github.io/atlas/goma-map.html


In [11]:
# Buildings as a population proxy
# (Where structures cluster, people cluster)

print(f"Querying OpenStreetMap for buildings in {AREA['name']}...")
buildings = ox.features.features_from_bbox(
    bbox=bbox_tuple,
    tags={'building': True}
)
print(f"\nReturned {len(buildings):,} building features")
print(f"Geometry types: {buildings.geometry.geom_type.value_counts().to_dict()}")

Querying OpenStreetMap for buildings in Goma (city and northern periphery)...

Returned 178,715 building features
Geometry types: {'Polygon': 178705, 'Point': 10}


In [12]:
# Convert to points (centroid)
buildings_proj = buildings.to_crs(epsg=3857)
buildings_proj['geometry'] = buildings_proj.geometry.centroid
buildings_pts = buildings_proj.to_crs(epsg=4326)

print(f"{len(buildings_pts):,} buildings, now as point geometries")

178,715 buildings, now as point geometries


In [13]:
# Spatial join: which hex contains each building?
b_joined = gpd.sjoin(buildings_pts[['geometry']], hexes, how='left', predicate='within')

# Count buildings per hex
b_counts = b_joined.groupby('h3_id').size().rename('building_count')

# Merge into our hex GeoDataFrame
hexes = hexes.merge(b_counts, on='h3_id', how='left')
hexes['building_count'] = hexes['building_count'].fillna(0).astype(int)

# Stats
print(f"Total hexes:                       {len(hexes)}")
print(f"Hexes with ≥1 building:            {(hexes['building_count'] > 0).sum()}")
print(f"Hexes with ≥50 buildings:          {(hexes['building_count'] >= 50).sum()}")
print(f"Hexes with ≥200 buildings:         {(hexes['building_count'] >= 200).sum()}")
print(f"Max buildings in a single hex:     {hexes['building_count'].max():,}")
print(f"Median building count (where >0):  {int(hexes[hexes['building_count']>0]['building_count'].median())}")

Total hexes:                       212
Hexes with ≥1 building:            160
Hexes with ≥50 buildings:          121
Hexes with ≥200 buildings:         93
Max buildings in a single hex:     5,287
Median building count (where >0):  447


In [14]:
# The portfolio money shot:
# How many hexes have lots of buildings but zero facilities?

dense_no_service = hexes[(hexes['building_count'] >= 50) & (hexes['facility_count'] == 0)]
dense_with_service = hexes[(hexes['building_count'] >= 50) & (hexes['facility_count'] > 0)]

print(f"Hexes with ≥50 buildings AND 0 facilities:  {len(dense_no_service)}")
print(f"Hexes with ≥50 buildings AND ≥1 facility:   {len(dense_with_service)}")
print(f"\nIn other words: of all the densely-populated areas in this bbox,")
print(f"{len(dense_no_service)} out of {len(dense_no_service)+len(dense_with_service)} have no health facility nearby.")

Hexes with ≥50 buildings AND 0 facilities:  62
Hexes with ≥50 buildings AND ≥1 facility:   59

In other words: of all the densely-populated areas in this bbox,
62 out of 121 have no health facility nearby.


## The gap

OSM building footprints serve as the population proxy. In Goma this is imperfect for two specific reasons. First, informal structures and shelter tarps in IDP camps may not be mapped as buildings. Second, some buildings recorded in OSM may have been destroyed by the 2021 lava flows but remain in the historical record. The proxy is rough, but it still identifies where dense built fabric concentrates.

We classify each 0.5 km hex into one of three categories:

- **Dense, no facility** — building count ≥ 50, facility count = 0. The gap zones.
- **Dense, served** — building count ≥ 50, facility count ≥ 1. Adequately reached.
- **Low density** — fewer than 50 mapped buildings. Outside the dense urban footprint, including agricultural land, lava fields, and unmapped informal areas.

In [15]:
# Classify each hex into one of three categories
def classify_hex(row):
    if row['building_count'] < 50:
        return 'low_density'        # rural / outside camp / empty
    elif row['facility_count'] == 0:
        return 'underserved'        # dense but no facility
    else:
        return 'served'             # dense and at least 1 facility

hexes['access_class'] = hexes.apply(classify_hex, axis=1)

# Summary
print("Hexes by access class:")
print(hexes['access_class'].value_counts())

print(f"\nBuildings in 'underserved' hexes:  "
      f"{hexes[hexes['access_class']=='underserved']['building_count'].sum():>7,}")
print(f"Buildings in 'served' hexes:       "
      f"{hexes[hexes['access_class']=='served']['building_count'].sum():>7,}")

# The headline number
total_dense = hexes[hexes['building_count'] >= 50]['building_count'].sum()
underserved = hexes[hexes['access_class']=='underserved']['building_count'].sum()
pct = 100 * underserved / total_dense
print(f"\n{pct:.1f}% of buildings in dense areas are in 'underserved' hexes.")

Hexes by access class:
access_class
low_density    91
underserved    62
served         59
Name: count, dtype: int64

Buildings in 'underserved' hexes:   38,373
Buildings in 'served' hexes:       138,329

21.7% of buildings in dense areas are in 'underserved' hexes.


In [16]:
# Service-population alignment map
m3 = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles='OpenStreetMap'
)

# Color by access class — red where the gap is
def gap_style_fn(feature):
    cls = feature['properties']['access_class']
    if cls == 'low_density':
        return {'fillColor': 'transparent', 'color': '#cccccc',
                'weight': 0.3, 'fillOpacity': 0.0}
    elif cls == 'underserved':
        return {'fillColor': '#cb181d', 'color': '#67000d',
                'weight': 0.6, 'fillOpacity': 0.65}
    else:  # served
        return {'fillColor': '#41ab5d', 'color': '#005a32',
                'weight': 0.6, 'fillOpacity': 0.55}

folium.GeoJson(
    hexes.to_json(),
    style_function=gap_style_fn,
    tooltip=folium.GeoJsonTooltip(
        fields=['building_count', 'facility_count', 'access_class'],
        aliases=['Buildings:', 'Health facilities:', 'Status:']
    )
).add_to(m3)

# Facility points on top
for _, row in facilities_pts.iterrows():
    name = (row.get('name:en') or row.get('name')
            or str(row.get('amenity', 'Health facility')))
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4, color='#222', weight=1.5,
        fill=True, fillColor='white', fillOpacity=1.0,
        tooltip=name
    ).add_to(m3)

# Legend
gap_legend = '''
<div style="position: fixed; bottom: 30px; left: 30px;
            background-color: white; border: 1px solid #999;
            padding: 10px 14px; font-family: Arial, sans-serif;
            font-size: 12px; z-index: 9999;
            box-shadow: 0 2px 6px rgba(0,0,0,0.15); line-height: 1.6;
            max-width: 240px;">
    <div style="font-weight: bold; margin-bottom: 6px;">
        Service-population alignment
    </div>
    <div><span style="background:#cb181d;opacity:0.65;width:18px;height:14px;
                      display:inline-block;vertical-align:middle;margin-right:6px;"></span>
         Dense, no health facility</div>
    <div><span style="background:#41ab5d;opacity:0.55;width:18px;height:14px;
                      display:inline-block;vertical-align:middle;margin-right:6px;"></span>
         Dense, served by 1+ facility</div>
    <div><span style="background:white;border:1px solid #ccc;width:18px;height:14px;
                      display:inline-block;vertical-align:middle;margin-right:6px;"></span>
         Low density (&lt;50 buildings)</div>
    <div style="border-top:1px solid #eee;margin-top:6px;padding-top:4px;font-size:11px;color:#666;">
        Hex resolution: H3 res 8 (~0.5 km)<br>
        Buildings used as population proxy<br>
        Sources: OpenStreetMap, HOTOSM
    </div>
</div>
'''
m3.get_root().html.add_child(folium.Element(gap_legend))

# Save it
gap_path = PROJECT_ROOT / "atlas" / "goma-gap-map.html"
m3.save(str(gap_path))
print(f"Gap map saved to: {gap_path}")

m3

Gap map saved to: /Users/sajjadsharifi/UrbanProjects/heissj.github.io/atlas/goma-gap-map.html


## What the map shows

Goma's pattern adds a new dimension to the atlas. The city proper, near the lake, contains most of the formal health infrastructure, while the northern periphery, where IDP camps cluster and where new displacement waves have settled, shows a higher gap density. The 2021 lava-flow corridor cuts through the city's northern districts and likely shapes both the location of remaining built fabric and the location of post-eruption rebuilding.

Across the four cities in this atlas, the gap patterns trace different stories. In Cox's Bazar the gap is at the camp periphery. In Kabul the gap is everywhere outside a few service clusters. In Mosul the gap follows the recovery divide across the Tigris. In Goma the gap reflects layered displacement, both from the volcano and from conflict to the north. Same methodology, four very different fragility stories.

### Limitations

- **OSM coverage of IDP camps is uneven.** Settlements that have emerged since 2022 may be underrepresented, which means the population in some hexes is undercounted.
- **The volcano matters but is not in this map.** Nyiragongo's hazard zones, the 2021 lava flow corridors, and ongoing seismic risk are not layered onto this analysis. Adding them would be the next natural step.
- **Conflict displacement is dynamic.** IDP populations around Goma have moved repeatedly between 2022 and 2025. Any spatial map of services is a snapshot, not a stable picture.
- **OSM facility tagging is partial.** Many small clinics in the IDP camps are operated by humanitarian agencies and not tagged in OSM.

### Methodology

OpenStreetMap features pulled via `osmnx`. Hexagonal tiling via Uber's H3 (resolution 8). Spatial joins and aggregation in `geopandas`. Maps rendered with `folium`. All code is visible in the cells above (click "Show code" on any block).